# A Process Refuses to Terminate

**Scenario:** A production AI/ML process (FastAPI -> Model Service -> Python Worker -> GPU/CPU) won't die after `kill` or even `kill -9`. This notebook is a hands-on, self-contained lab for diagnosing *why*, instead of blindly escalating signals.

> Run the `!` shell cells in order. Each section builds the diagnostic picture: signal behavior -> process tree -> process state -> root cause (kernel/I-O, zombie, or supervisor-restart).

**Learning objectives**
- Linux signals: `SIGTERM` vs `SIGKILL` vs `SIGINT`
- Process states: `R`, `S`, `D` (uninterruptible sleep), `T`, `Z` (zombie)
- Parent/child relationships and process trees
- Why `kill -9` sometimes does nothing
- A repeatable investigation workflow for stuck AI/ML workers (PyTorch jobs, LLM inference servers, Celery/Ray/Kafka workers, GPU workers)


## 1. Create a Dummy AI/ML Worker

We simulate a model worker that **intentionally ignores `SIGTERM`** - a common bug pattern in real inference services (e.g., a custom signal handler that doesn't call `sys.exit()`).

In [1]:
%%writefile stubborn_model.py
import signal
import time
import os

def ignore_sigterm(signum, frame):
    print("SIGTERM received, but I am ignoring it!")

signal.signal(signal.SIGTERM, ignore_sigterm)

print(f"AI Model Worker Started")
print(f"PID: {os.getpid()}")

while True:
    print("Model inference worker is running...")
    time.sleep(5)


Writing stubborn_model.py


Launch it in the background so this notebook stays interactive, and capture its PID.

In [29]:
# !nohup python3 stubborn_model.py > worker.log 2>&1 &
# !sleep 1
# !pgrep -f stubborn_model.py

import subprocess

subprocess.Popen(
    ["wsl", "nohup", "python3", "stubborn_model.py"],
    stdout=open("worker.log", "a"),
    stderr=subprocess.STDOUT,
)


<Popen: returncode: None args: ['wsl', 'nohup', 'python3', 'stubborn_model.py']>

Set `PID` below to the value printed above so the shell env var `$PID` is reusable in later commands.

In [5]:
import subprocess
import os
PID = subprocess.check_output(["wsl","pgrep", "-f", "stubborn_model.py"]).decode().split()[0]
os.environ["PID"] = PID
print("PID =", PID)


PID = 356


## 2. Find and Inspect the Process

In [12]:
# !ps aux | grep stubborn_model

import subprocess

result = subprocess.run(
    ["wsl", "ps", "aux"],
    capture_output=True,
    text=True
)

lines = result.stdout.splitlines()

if lines:
    header = lines[0]  # The header row: USER PID %CPU %MEM ...
    matches = [line for line in lines[1:] if "stubborn_model" in line]
    
    print(header)
    print("\n".join(matches))

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
usama        356  0.0  0.0  13532  8440 pts/0    Ss+  10:48   0:00 python3 stubborn_model.py
usama       2067  0.0  0.0  13560  8492 pts/2    Ss+  10:55   0:00 python3 -u stubborn_model.py


In [ ]:
!ps -fp 2067

UID          PID    PPID  C STIME TTY          TIME CMD
usama       2067    2062  0 10:55 pts/2    00:00:00 python3 -u stubborn_model.py


**Key fields**

| Field | Meaning |
|---|---|
| `PID`  | Process ID |
| `PPID` | Parent Process ID |


## 3. Attempt Graceful Termination (`SIGTERM`)

`kill PID` is shorthand for `kill -15 PID`. It **requests** termination - the app can handle it, ignore it, or exit.

In [ ]:
!kill 2067

In [ ]:
!tail -5 worker.log

Model inference worker is running...
Model inference worker is running...
Model inference worker is running...
Model inference worker is running...
Model inference worker is running...


In [ ]:
!ps -p 2067

    PID TTY          TIME CMD
   2067 pts/2    00:00:00 python3


**Result:** the process is still alive - the log shows `"SIGTERM received, but I am ignoring it!"`.

```
kill PID
   |
   v
SIGTERM --> app can: handle gracefully | ignore | exit
```
SIGTERM is a *request*, never a *guarantee*.

## 4. Escalate to `SIGKILL`

`SIGKILL` cannot be caught, blocked, or ignored - the **kernel**, not the app, terminates the process.

In [ ]:
!kill -9 2067

In [ ]:
!ps -p 2067

    PID TTY          TIME CMD


| Signal | Meaning | App can handle it? |
|---|---|---|
| `SIGTERM` (15) | Please terminate | Yes |
| `SIGINT` (2)   | Interrupt (Ctrl+C) | Yes |
| `SIGKILL` (9)  | Terminate immediately | **No** |

Normally the process disappears here. If it *still* shows up, move to the next sections - the cause is no longer "the app ignored a signal."


## 5. Why a Production AI Worker Might Resist Termination

```
FastAPI -> Inference Worker -> PyTorch / Transformers -> GPU -> Model
```

Common root causes when `kill` / `kill -9` don't work:
1. Long-running inference request in progress
2. Blocking GPU operation
3. Deadlock
4. Infinite loop
5. Buggy signal handler (as simulated above)
6. Background/child worker still active
7. A supervisor (systemd, Docker, Kubernetes) restarting the worker


## 6. Scenario: A Supervisor Restarts the Worker

```
systemd(1) -> ai-service(2000) -> python(3456)
```
Killing `python(3456)` looks successful, then the process **reappears** because `ai-service` detected the child's death and relaunched it. Inspect the parent, not just the PID you killed.

In [ ]:
!ps -o pid,ppid,cmd -p $PID

    PID    PPID CMD


Look up and inspect the parent PID directly:

In [ ]:
!PPID=$(ps -o ppid= -p $PID | tr -d " ")
!echo "Parent PID: $PPID"
!ps -fp $PPID

## 7. Visualize the Full Process Tree

A single worker is often one of several children under a supervisor (Docker/Ray/Celery). Killing one may not fix the underlying issue.

In [ ]:
!pstree -p $PID

```
systemd -> Docker -> Model Server -> Worker 1
                                   -> Worker 2
                                   -> Worker 3
```


## 8. Check the Process State

| State | Meaning |
|---|---|
| `R` | Running |
| `S` | Sleeping |
| `D` | Uninterruptible sleep (usually I/O) |
| `T` | Stopped |
| `Z` | Zombie |


In [ ]:
!ps -o pid,state,stat,cmd -p $PID

## 9. The `D` State Trap

A process in **uninterruptible sleep (`D`)** is blocked on a kernel operation - disk I/O, a network filesystem, or a stalled storage device. `kill -9` **queues** the kill; it only takes effect once the process returns from that kernel call.

```
SIGKILL -> kernel marks process for termination
        -> process must first return from the kernel operation
        -> then termination happens
```

**AI/ML example:** `model = load_model("/mnt/network-storage/model")` hangs because the network mount froze. The Python process enters `D` state - the real problem is **storage**, not the model code.

## 10. Inspect What the Process Is Waiting On

In [ ]:
!cat /proc/$PID/status | head -20

In [ ]:
!echo -n "waiting on: "; cat /proc/$PID/wchan; echo

`wchan` names the kernel function the process is blocked in - a strong clue for I/O vs. GPU vs. lock contention.

## 11. Check Open Files

Look for model checkpoints, logs, database sockets, or mounted storage the process might be blocked on.

In [ ]:
!lsof -p $PID

## 12. Check Network Dependencies

```
FastAPI -> Inference Worker -> Redis / PostgreSQL / Vector DB
```
If a downstream dependency hangs, the worker can appear "stuck" while actually blocked on a socket read.

In [ ]:
!lsof -i -p $PID

In [ ]:
!sudo ss -ltnp

## 13. Zombie Processes

```
Worker exits -> parent has not called wait() -> child stays as a zombie (state Z)
```
**You cannot `kill -9` a zombie** - it has already terminated; only its exit-status entry remains. Fix the **parent**.

In [ ]:
!ps aux | awk '$8 ~ /Z/'

In [ ]:
!ps -eo pid,ppid,state,cmd | awk '$3 == "Z"'

**AI/ML example:** a Training Manager spawns Data / GPU / Evaluation workers. The Evaluation Worker exits, but the manager never reaps it via `wait()` -> it lingers as a zombie. Find its `PPID` and inspect the manager process, e.g. `ps -o pid,ppid,state,cmd -p <ZOMBIE_PID>`.

## 14. Investigation Workflow (Decision Tree)

```
Process refuses to terminate
        |
        v
   Check state
        |
   -----+--------------
   R/S       Z            D
   |         |            |
SIGTERM  Investigate  Investigate
   |      the parent   I/O, kernel,
SIGKILL                 storage
```


## 15. Quick Command Reference

In [ ]:
!echo "Find process:      ps aux | grep PROCESS_NAME"
!echo "Inspect process:   ps -fp PID"
!echo "Check state:       ps -o pid,ppid,state,stat,cmd -p PID"
!echo "Send SIGTERM:      kill PID"
!echo "Send SIGKILL:      kill -9 PID"
!echo "Process tree:      pstree -p PID"
!echo "Inspect /proc:     cat /proc/PID/status"
!echo "Kernel wait chan:  cat /proc/PID/wchan"
!echo "Open files:        lsof -p PID"

## 16. Final Lab Challenge

Given:
```
FastAPI Server -> Inference Worker -> Large Language Model
```
`kill PID` and `kill -9 PID` both fail to remove the process. Run the full diagnostic sequence below, then classify the problem as an **application**, **parent-process**, **zombie**, or **kernel/I-O** issue.

In [ ]:
!ps -o pid,ppid,state,stat,cmd -p $PID
!pstree -p $PID
!cat /proc/$PID/status
!cat /proc/$PID/wchan; echo
!lsof -p $PID

**Your diagnosis:** _(fill in after reviewing the output above)_
- [ ] Application problem (bad/ignored signal handler)
- [ ] Parent-process problem (supervisor keeps restarting it)
- [ ] Zombie problem (parent isn't reaping exit status)
- [ ] Kernel/I-O problem (stuck in `D` state on storage/network)

## 17. Cleanup

In [ ]:
!kill -9 $PID 2>/dev/null; rm -f stubborn_model.py worker.log; echo "cleaned up"

## Key Takeaway

A process that refuses to terminate is usually a **symptom**, not the root cause. Before reaching for `kill -9` on everything:

```
Stuck worker -> what state is it in? -> who is the parent? ->
what resource is it waiting on? -> is something restarting it? -> root cause
```

This diagnostic habit - checking state, parent, and wait channel before escalating signals - is a core skill for AI infrastructure, model-serving and MLOps, where "stuck" processes are frequently GPU, storage, deadlock, or orchestrator (Docker/Kubernetes) issues rather than broken model code.